# Moon environmental EDA — native support and joint availability

Ported from `notebooks/exploration/25_environment_native_support_eda.ipynb`. Original source files are unchanged. Inputs and all generated data stay under `darrel stuff/data/`.

Run local notebooks 05, 06 and 13 first. Restore the original native-layer unit policy under `darrel stuff/data/reference/environment-readiness-20261028/`. The illumination and Diviner analysis cells below are copied without changing their calculations, thresholds or cohorts. The Earth climate and OSM inventory sections are excluded from this Moon notebook.

**Code copied first; not executed locally.** No historical outputs are presented as new results. Install the original processing dependencies and read `../../EDA_Notes.md` before Run All.


In [1]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "outline.md").is_file() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository or one of its subfolders")
DATA_ROOT = ROOT / "darrel stuff/data"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "darrel stuff/EDA"))
from environment_workflow import LocalRun as Run, latest_completed_run, select_climate_manifest

print("Source data:", DATA_ROOT)
print("Processed outputs:", DATA_ROOT / "processed")


Source data: c:\Darrel\Self-Projects\1 group proj\NASA-Space-Apps-Challenge-2026\darrel stuff\data
Processed outputs: c:\Darrel\Self-Projects\1 group proj\NASA-Space-Apps-Challenge-2026\darrel stuff\data\processed


## Accepted local inputs and original unit policy

In [2]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from IPython.display import display
from pyproj import Transformer
from rasterio.crs import CRS
from rasterio.windows import Window
from notebooks.exploration.environment_analysis import diviner_cell_matrix
from notebooks.shared.inspection import check_input, file_hash, verify_run
from notebooks.shared.processing import MOON_CRS, MOON_GEOGRAPHIC

MOON_RUN_ID = None
THERMAL_RUN_ID = None
VIEWS_RUN_ID = None
# Restore the original unit policy; never infer documented units from a filename.
UNIT_POLICY = DATA_ROOT / "reference/environment-readiness-20261028/native_layer_units.json"
if not UNIT_POLICY.is_file():
    raise FileNotFoundError(f"Restore the original native-layer unit policy before analysis: {UNIT_POLICY}")
for stage, run_id in [("lunar_terrain_environment", MOON_RUN_ID),
                      ("diviner_temperature", THERMAL_RUN_ID), ("lunar_terrain_views", VIEWS_RUN_ID)]:
    latest_completed_run(ROOT, stage, run_id)
ANALYSIS = Run(ROOT, "analysis/moon_environment_eda", {
    "source": "notebooks/exploration/25_environment_native_support_eda.ipynb",
    "scope": "Moon environmental sections; original Earth/OSM sections excluded",
    "moon_run_id": MOON_RUN_ID, "thermal_run_id": THERMAL_RUN_ID, "views_run_id": VIEWS_RUN_ID,
}, "Moon/Environmental_Context/25_environment_native_support_eda.ipynb")
ANALYSIS.__enter__()
OUT = ANALYSIS.directory
expected = {}
evidence = {}

def pin(path):
    evidence[str(path.relative_to(ROOT))] = file_hash(path)
    ANALYSIS.input(path)
    return path

def register(stage, run_id=None):
    folder = latest_completed_run(ROOT, stage, run_id)
    expected.update(verify_run(folder))
    pin(folder / "provenance.json")
    return folder

def checked(path):
    audit = {}
    check_input(path, expected, audit)
    evidence[str(path.relative_to(ROOT))] = audit[str(path.resolve())]
    ANALYSIS.input(path)
    return path

def savefig(fig, name):
    fig.savefig(OUT / (name + ".png"), dpi=150, bbox_inches="tight")
    fig.savefig(OUT / (name + ".pdf"), bbox_inches="tight")
    display(fig)
    plt.close(fig)

def short(region):
    return region.split("_", 2)[-1].replace("_", " ")

moon_run = register("lunar_terrain_environment", MOON_RUN_ID)
thermal_run = register("diviner_temperature", THERMAL_RUN_ID)
views_run = register("lunar_terrain_views", VIEWS_RUN_ID)
for path in [UNIT_POLICY, ROOT / "notebooks/exploration/25_environment_native_support_eda.ipynb",
             ROOT / "notebooks/exploration/environment_analysis.py", ROOT / "notebooks/shared/processing.py"]:
    pin(path)
units = pd.read_json(UNIT_POLICY, encoding="utf-8-sig")
thermal_summary = pd.read_csv(checked(thermal_run / 'regional_bin_summary.csv'))
regions = sorted(thermal_summary.region.unique())
products = sorted(thermal_summary.source_product.unique())
presence = thermal_summary.pivot(index='region', columns='source_product', values='valid_temperature_rows').gt(0)
common_products = presence.columns[presence.all(axis=0)].tolist()
assert len(regions) == 7 and len(products) == 144 and len(common_products) == 17
pin(checked(thermal_run / 'interpretation.json'))
CONFIG = {"scope": "Moon development regions only", "common_products": common_products,
          "thermal_denominator": "Union of observed source centers, including invalid rows",
          "joint_rule": "Containing native pixel at each observed Diviner center; no interpolation",
          "physical_registration": "Unverified; technical coordinate consistency only"}
ANALYSIS.record["configuration"] = CONFIG


FileNotFoundError: Restore the original native-layer unit policy before analysis: c:\Darrel\Self-Projects\1 group proj\NASA-Space-Apps-Challenge-2026\darrel stuff\data\reference\environment-readiness-20261028\native_layer_units.json

## Native illumination and source/crop coordinate consistency

In [ ]:
layer_files = {'solar_visibility': 'AVGVISIB_85S_060M_201608.TIF',
               'earth_visibility': 'AVGVISIB_85S_060M_201608_EARTH.TIF',
               'permanent_shadow': 'LPSR_85S_060M_201608.TIF',
               'sky_view_sr': 'SKYV_65S_240M.TIF'}
illumination_rows = []
grid_checks = []
region_bounds = {}
illumination_paths = {}
moon_provenance = json.loads((moon_run / 'provenance.json').read_text())
raw_hashes = {str((ROOT / path).resolve()): item['sha256'] for path, item in moon_provenance['inputs'].items()}
for region in regions:
    quality = json.loads(checked(moon_run / 'moon' / region / 'quality.json').read_text())
    west, south, east, north = quality['bounds_m']
    region_bounds[region] = (west, south, east, north)
    for layer, filename in layer_files.items():
        path = moon_run / 'moon' / region / 'native/illumination' / filename
        illumination_paths[region, layer] = path
        unit = units.loc[(units.region == region) & (units.file == filename)].iloc[0]
        assert unit.status == 'documented'
        meta = next(item for item in quality['native_layers'] if Path(item['source']).name == filename)
        source_path = Path(meta['source'])
        if str(source_path.relative_to(ROOT)) not in evidence:
            assert file_hash(source_path) == raw_hashes[str(source_path.resolve())]
            pin(source_path)
        with rasterio.open(checked(path)) as src, rasterio.open(source_path) as original:
            assert src.crs == original.crs == CRS.from_string(MOON_CRS)
            assert src.scales == (1.0,) and src.offsets == (0.0,)
            col_offset, row_offset = (~original.transform) * (src.transform.c, src.transform.f)
            assert np.allclose([col_offset, row_offset], np.round([col_offset, row_offset]), atol=1e-7)
            window = Window.from_slices((round(row_offset), round(row_offset) + src.height),
                                        (round(col_offset), round(col_offset) + src.width))
            assert original.window_transform(window).almost_equals(src.transform)
            raw = original.read(1, window=window, masked=True).astype('float32').filled(np.nan)
            decoded = raw * original.scales[0] + original.offsets[0]
            values = src.read(1, masked=True).astype('float32').filled(np.nan)
            np.testing.assert_allclose(values, decoded, rtol=1e-6, atol=1e-6, equal_nan=True)
            rows, cols = np.indices(src.shape)
            x = src.transform.c + (cols + .5) * src.transform.a
            y = src.transform.f + (rows + .5) * src.transform.e
            inside = (x >= west) & (x < east) & (y >= south) & (y < north)
            finite = values[inside & np.isfinite(values)]
            q10, median, q90 = np.quantile(finite, [.1, .5, .9])
            illumination_rows.append({'region': region, 'layer': layer, 'unit': unit.unit,
                                      'native_spacing_m': src.res[0], 'inside_centers': int(inside.sum()),
                                      'valid_centers': len(finite), 'mean': float(finite.mean()),
                                      'q10': q10, 'median': median, 'q90': q90,
                                      'minimum': float(finite.min()), 'maximum': float(finite.max())})
            grid_checks.append({'region': region, 'layer': layer, 'source_window_col': round(col_offset),
                                'source_window_row': round(row_offset), 'shape': list(src.shape),
                                'transform': list(src.transform), 'crs_equal': True,
                                'decoded_source_window_matches': True})
illumination = pd.DataFrame(illumination_rows)
illumination.to_csv(OUT / 'lunar_illumination_statistics.csv', index=False)
(OUT / 'source_crop_checks.json').write_text(json.dumps(grid_checks, indent=2), encoding='utf-8')
fig, axes = plt.subplots(1, 4, figsize=(15, 5), sharey=True, layout='constrained')
for ax, layer in zip(axes, layer_files, strict=True):
    part = illumination.loc[illumination.layer == layer].set_index('region').loc[regions]
    y = np.arange(len(regions))
    if layer == 'permanent_shadow':
        ax.scatter(part['mean'], y, color='#176b72')
        ax.set_title('Permanent-shadow fraction\nMean of binary native class')
    else:
        ax.hlines(y, part.q10, part.q90, color='#176b72', linewidth=3)
        ax.scatter(part['median'], y, color='#d17b45', zorder=3)
        ax.set_title(layer.replace('_', ' ') + '\nNative-center median and 10–90%')
    ax.set_xlabel('Steradians' if layer == 'sky_view_sr' else 'Fraction')
    ax.set_yticks(y, [short(r) for r in regions])
    ax.grid(alpha=.2)
axes[0].invert_yaxis()
fig.suptitle('Lunar modeled context: original region rectangles\n60 m visibility/shadow and 240 m sky view; no duration or power inference')
savefig(fig, 'lunar_illumination')

## Diviner cell-level phase coverage and joint availability

In [ ]:
project = Transformer.from_crs(MOON_GEOGRAPHIC, MOON_CRS, always_xy=True)
channels = ['elevation_m', 'relative_elevation_m', 'slope_deg', 'laplacian_per_m', 'local_elevation_std_m']
thermal_rows = []
thermal_bin_rows = []
low_tail_rows = []
joint_rows = []
coverage_curves = {}


def sample_native(path, coordinates):
    with rasterio.open(checked(path)) as src:
        assert src.crs == CRS.from_string(MOON_CRS)
        samples = np.ma.vstack(list(src.sample(coordinates, masked=True))).astype(float).filled(np.nan)[:, 0]
    return samples


for region in regions:
    frames = [pd.read_csv(checked(p)) for p in sorted((thermal_run / 'moon' / region / 'native_240m').glob('*.csv.gz'))]
    assert len(frames) == 144
    observations = pd.concat([frame for frame in frames if not frame.empty], ignore_index=True)
    low_tail = observations.loc[observations.valid_temperature].nsmallest(10, 'tbol').copy()
    low_tail['region'] = region
    low_tail_rows.append(low_tail)
    matrix = diviner_cell_matrix(observations, products)
    centers = pd.DataFrame(observations[['clon', 'clat', 'x_m', 'y_m']]).drop_duplicates()
    assert not centers.duplicated(['clon', 'clat']).any()
    centers = centers.set_index(['clon', 'clat']).loc[matrix.index].reset_index()
    x, y = project.transform(centers.clon.to_numpy(), centers.clat.to_numpy())
    assert np.allclose(x, centers.x_m, atol=1e-6, rtol=0) and np.allclose(y, centers.y_m, atol=1e-6, rtol=0)
    west, south, east, north = region_bounds[region]
    assert ((x >= west) & (x < east) & (y >= south) & (y < north)).all()
    valid = matrix.notna()
    all_counts = np.asarray(valid.sum(axis=1).to_numpy())
    common_counts = np.asarray(valid[common_products].sum(axis=1).to_numpy())
    complete_common = common_counts == len(common_products)
    centers['available_products_of_144'] = all_counts
    centers['available_common_products_of_17'] = common_counts
    coords = list(zip(x, y, strict=True))
    for layer in layer_files:
        centers[layer] = sample_native(illumination_paths[region, layer], coords)
    illumination_available = np.isfinite(centers[list(layer_files)].to_numpy()).all(axis=1)
    centers['all_four_illumination_layers_available'] = illumination_available
    counts = {'region': region, 'observed_center_union': len(matrix),
              'region_products_with_any_valid_cell': int(np.asarray(valid.to_numpy()).any(axis=0).sum()),
              'centers_with_any_valid_temperature': int((all_counts > 0).sum()),
              'centers_complete_144': int((all_counts == 144).sum()),
              'centers_complete_common_17': int(complete_common.sum()),
              'common_17_complete_fraction': float(complete_common.mean()),
              'median_available_products': float(np.median(all_counts)),
              'minimum_available_products': int(all_counts.min()), 'maximum_available_products': int(all_counts.max())}
    thermal_rows.append(counts)
    coverage_curves[region] = all_counts
    for product in products:
        values = matrix[product].dropna().to_numpy()
        paired = matrix.loc[complete_common, product].dropna().to_numpy() if product in common_products else np.array([])
        thermal_bin_rows.append({'region': region, 'source_product': product, 'valid_centers': len(values),
                                 'observed_center_union': len(matrix),
                                 'fraction_of_observed_centers': len(values) / len(matrix),
                                 'spatial_median_bin_average_K': float(np.median(values)) if len(values) else np.nan,
                                 'spatial_q10_bin_average_K': float(np.quantile(values, .1)) if len(values) else np.nan,
                                 'spatial_q90_bin_average_K': float(np.quantile(values, .9)) if len(values) else np.nan,
                                 'complete_common_cohort_median_K': float(np.median(paired)) if len(paired) else np.nan})
    for view in ['full_context', 'image_supported']:
        support = np.ones(len(matrix), dtype=bool)
        for channel in channels:
            support &= np.isfinite(sample_native(views_run / 'moon' / region / view / (channel + '.tif'), coords))
        centers[view + '_complete_terrain_at_center'] = support
        joint = support & illumination_available & complete_common
        joint_rows.append({'region': region, 'terrain_view': view, 'observed_diviner_centers': len(matrix),
                           'terrain_complete_at_center': int(support.sum()),
                           'illumination_complete_at_center': int(illumination_available.sum()),
                           'common_17_thermal_complete': int(complete_common.sum()),
                           'joint_complete_at_center': int(joint.sum()),
                           'fraction_of_terrain_supported_centers': float(joint.sum() / support.sum()) if support.any() else np.nan})
    centers.to_csv(OUT / f'diviner_centers_{region}.csv.gz', index=False, compression={'method': 'gzip', 'mtime': 0})
    print(short(region), counts)
    del observations, frames, matrix
thermal = pd.DataFrame(thermal_rows)
thermal_bins = pd.DataFrame(thermal_bin_rows)
joint_table = pd.DataFrame(joint_rows)
thermal.to_csv(OUT / 'diviner_cell_coverage.csv', index=False)
thermal_bins.to_csv(OUT / 'diviner_bin_distributions.csv', index=False)
joint_table.to_csv(OUT / 'lunar_joint_availability.csv', index=False)
pd.concat(low_tail_rows, ignore_index=True).to_csv(OUT / 'diviner_low_tail_examples.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(13, 5), layout='constrained')
for region, counts in coverage_curves.items():
    ordered = np.sort(counts)
    axes[0].plot(ordered, np.arange(1, len(ordered) + 1) / len(ordered), label=short(region))
axes[0].set(xlabel='Valid products per observed native center (of 144)', ylabel='Cumulative fraction of observed centers', xlim=(0, 144), ylim=(0, 1))
axes[0].legend(fontsize=7)
axes[0].grid(alpha=.2)
fraction_matrix = thermal_bins.pivot(index='region', columns='source_product', values='fraction_of_observed_centers').loc[regions, products]
im = axes[1].imshow(fraction_matrix, aspect='auto', vmin=0, vmax=1, cmap='viridis')
axes[1].set_yticks(range(len(regions)), [short(r) for r in regions], fontsize=8)
axes[1].set_xticks([12 + 24*i for i in range(6)], [f'Ls{v:03d}' for v in [30, 90, 150, 210, 270, 330]])
axes[1].set_xlabel('144 phase products: 24 subsolar bins within each Ls group')
fig.colorbar(im, ax=axes[1], label='Fraction of observed centers with valid temperature', shrink=.8)
fig.suptitle('Diviner coverage is spatially incomplete within phase bins\nDenominator: union of observed source centers; not all possible cells in each rectangle')
savefig(fig, 'diviner_cell_coverage')
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True, layout='constrained')
for region in regions:
    part = thermal_bins.loc[(thermal_bins.region == region) & thermal_bins.source_product.isin(common_products)].set_index('source_product').loc[common_products]
    axes[0].scatter(range(17), part.spatial_median_bin_average_K, s=16, label=short(region))
    axes[1].scatter(range(17), part.complete_common_cohort_median_K, s=16, label=short(region))
for ax, title in zip(axes, ['Each bin uses its own available cells', 'Fixed cells complete in all 17 common products'], strict=True):
    ax.set(title=title, xlabel='Common phase product: Ls center / subsolar bin', ylabel='Spatial median of bin-average brightness temperature (K)')
    ax.set_xticks(range(17), [p.split('_ls')[1].replace('_slon', '/').replace('_240', '') for p in common_products], rotation=65, fontsize=7)
    ax.grid(alpha=.2)
axes[0].legend(fontsize=7)
fig.suptitle('Diviner descriptive comparison on common phase products\nThree regions have no complete 17-product cohort; these are phase averages, not chronological temperatures')
savefig(fig, 'diviner_common_phase_temperature')
fig, ax = plt.subplots(figsize=(10, 5), layout='constrained')
for offset, view, color in [(-.18, 'full_context', '#176b72'), (.18, 'image_supported', '#d17b45')]:
    part = joint_table.loc[joint_table.terrain_view == view].set_index('region').loc[regions]
    ax.barh(np.arange(7) + offset, part.fraction_of_terrain_supported_centers * 100, height=.32, label=view.replace('_', ' '), color=color)
ax.set_yticks(np.arange(7), [short(r) for r in regions])
ax.invert_yaxis()
ax.set(xlabel='Percent of terrain-supported observed Diviner centers', xlim=(0, 100),
       title='Joint availability: all 17 common thermal products + four illumination layers\nContextual center lookup only; not full terrain-area or footprint coverage')
ax.legend()
savefig(fig, 'lunar_joint_availability')

## Interpretation

Keep missing phase bins and the observed-center denominator explicit. A complete-common-products cohort is a selected population, not a silent default filter. Solar visibility, Earth visibility, shadow class and sky view are separate quantities. None of these results establish thermal safety, continuous sunlight, communication reliability or equipment suitability.

Run the final cell only after all analysis cells succeed.

In [ ]:
assert len(regions) == 7 and len(products) == 144 and len(common_products) == 17
assert len(thermal) == 7 and len(thermal_bins) == 7 * 144 and len(joint_table) == 14
required = ["lunar_illumination_statistics.csv", "source_crop_checks.json", "diviner_cell_coverage.csv",
            "diviner_bin_distributions.csv", "lunar_joint_availability.csv", "diviner_low_tail_examples.csv"]
for name in required:
    if not (OUT / name).is_file():
        raise FileNotFoundError(f"Run all analysis cells before finalizing: {name}")
ANALYSIS.record["interpretation"] = {
    "thermal": "Phase-bin average surface brightness temperature; not chronology or mission extremes",
    "illumination": "Native modeled averages/classes; no continuous-darkness or delivered-power inference",
    "support": "Contextual native-center lookup, not full footprints or all terrain-area coverage",
    "quality": "Original unit policy required; missingness and incomplete cohorts remain explicit",
}
ANALYSIS.__exit__(None, None, None)
print("Completed local lunar environmental analysis:", OUT)
